# Incremental Landslide Data Extraction from GEE to HuggingFace

**Purpose:** Downloads landslide incident imagery (Sentinel-2, Sentinel-1, DEM) from
Google Earth Engine and uploads it to a HuggingFace dataset repository.

**Key improvement over previous versions:** This notebook is **idempotent** — it first
queries the HuggingFace repo for already-uploaded incidents, then downloads and uploads
*only* the missing ones. This avoids redundant downloads/uploads on re-runs and makes
it safe to run periodically as new incidents are added to the CSV.

**Workflow (landslide_workflow.md Stage 0):**
1. Queries HuggingFace for existing incident folders (`incident_<ID>/`)
2. Reads the landslide incidents CSV from Kaggle input
3. Computes the set difference: incidents in CSV but not on HuggingFace
4. For each missing incident:
   - Picks the least-cloudy single-date Sentinel-2 pre/post scene (not a median composite)
   - Downloads all 12 S2 bands + SCL, DEM slope/aspect, and paired Sentinel-1 GRD (VV/VH)
5. Uploads new incidents in batches of `upload_batch` to HuggingFace

## Why single-date instead of a median composite?

A median composite over an 18-month window blends hundreds of scenes into a flat,
cartoon-like image that smears away the actual post-event scene. Instead we:

1. Pick the **best single acquisition date** (least cloudy) closest to the incident
   on the pre- and post-event side.
2. Spatially mosaic neighbouring MGRS tiles from the same overpass day (a spatial
   stitch, NOT a temporal composite) so the scene looks continuous and real.
3. Download all 12 S2 reflectance bands + SCL so indices (NDVI, NDWI, BSI, NBR)
   can be computed later, plus DEM slope & aspect.
4. Optionally pull paired pre/post Sentinel-1 GRD (VV/VH, same orbit direction)
   for SAR amplitude-ratio change cue.

**Folder structure on HuggingFace:**
```
incident_<ID>/
  incident_<ID>_before.tif      (13-band S2: B1..B12, SCL)
  incident_<ID>_after.tif       (13-band S2)
  incident_<ID>_slope.tif       (1-band DEM slope @ 30m)
  incident_<ID>_aspect.tif      (1-band DEM aspect @ 30m)
  incident_<ID>_sar_pre.tif     (2-band VV/VH @ 10m, optional)
  incident_<ID>_sar_post.tif    (2-band VV/VH @ 10m, optional)
```

In [1]:
# %% [code]
# --------------------------------------------------------------------
# Imports
# --------------------------------------------------------------------
import os, glob, shutil, re
import pandas as pd
import ee
import time
import requests
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient
from concurrent.futures import ThreadPoolExecutor, as_completed

# --------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------
project = "landslide-identification-nepal"     # GEE project name
input_csv = "/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv"
upload_batch = 100                              # upload to HF every N incidents
repo_id = "sasudo2/landslides"                 # HF target dataset repo
DATASET_REVISION = "main"                      # branch/revision on HF
DOWNLOAD_DIR = '/kaggle/working/downloads'     # temp storage for GeoTIFFs
MAX_AOI_DEG = 0.1                              # max AOI extent in degrees
MAX_WORKERS = 2                                 # GEE parallel workers (keep low to avoid 429)

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# --------------------------------------------------------------------
# Authenticate: HuggingFace
# --------------------------------------------------------------------
user_secrets = UserSecretsClient()
huggingface_key = user_secrets.get_secret("huggingface_token")
api = HfApi(token=huggingface_key)
api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
print(f"HuggingFace repo '{repo_id}' ready.")

# --------------------------------------------------------------------
# Authenticate: Google Earth Engine
# --------------------------------------------------------------------
gee_key_path = "/kaggle/input/datasets/sanjayashrestha123/gee-key/landslide-identification-nepal-cccd90850069.json"
service_account = 'kaggle-import@landslide-identification-nepal.iam.gserviceaccount.com'
credentials = ee.ServiceAccountCredentials(service_account, gee_key_path)
try:
    ee.Initialize(credentials, project=project)
    print("Google Earth Engine initialized.")
except Exception as e:
    print("EE initialization failed.")
    raise e

# --------------------------------------------------------------------
# Load the landslide incidents CSV
# --------------------------------------------------------------------
df = pd.read_csv(input_csv)
df = df.iloc[721:]
df['incident_on'] = pd.to_datetime(df['incident_on'], dayfirst=True)
print(f"Loaded {len(df)} incidents from CSV.")

HuggingFace repo 'sasudo2/landslides' ready.


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Google Earth Engine initialized.
Loaded 3417 incidents from CSV.


## Step 1: Determine which incidents are already on HuggingFace

We query the HF dataset repo for all files, parse the folder names to extract
incident IDs, and compute the set of IDs present in the CSV but missing from HF.
Only those missing incidents will be downloaded from GEE and uploaded.

In [2]:
# %% [code]
print("\n=== Querying HuggingFace for existing incidents ===")

hf_ids = set()
try:
    repo_files = api.list_repo_files(repo_id=repo_id, repo_type="dataset", revision=DATASET_REVISION)
    for fpath in repo_files:
        m = re.match(r'incident_(\d+)/', fpath)
        if m:
            hf_ids.add(int(m.group(1)))
    print(f"Found {len(hf_ids)} existing incident folders on HuggingFace.")
except Exception as e:
    print(f"Could not list repo files (first run / empty repo?): {e}")
    print("Proceeding to download all incidents from CSV.")

csv_ids = set(df['id'].astype(int).tolist())
missing_ids = sorted(csv_ids - hf_ids)

print(f"Total incidents in CSV: {len(csv_ids)}")
print(f"Already on HuggingFace: {len(hf_ids)}")
print(f"Missing (will download): {len(missing_ids)}")
if missing_ids:
    print(f"First 5 missing IDs: {missing_ids[:5]}")


=== Querying HuggingFace for existing incidents ===
Found 1816 existing incident folders on HuggingFace.
Total incidents in CSV: 3417
Already on HuggingFace: 1816
Missing (will download): 1601
First 5 missing IDs: [37370, 37656, 37722, 37814, 38007]


## Step 2: GEE download helper functions

These functions handle cloud masking, AOI clamping, image downloading with retry
logic, best-scene selection, and the full per-incident export pipeline.

In [3]:
# %% [code]
def mask_s2_clouds(image):
    """Mask out cloud, shadow, and unclassified pixels using SCL band.
    Keeps: 2=dark, 4=vegetation, 5=not-vegetated, 6=water,
          7=unclassified, 11=snow. Drops cloud (8,9) and shadow (3).
    """
    scl = image.select('SCL')
    clean_mask = (scl.eq(2).bitwiseOr(scl.eq(4))
                           .bitwiseOr(scl.eq(5))
                           .bitwiseOr(scl.eq(6))
                           .bitwiseOr(scl.eq(7))
                           .bitwiseOr(scl.eq(11)))
    return image.updateMask(clean_mask)


def clamp_aoi(min_lon, min_lat, max_lon, max_lat):
    """Clamp AOI extent to MAX_AOI_DEG if either dimension exceeds it.
    Keeps original if both dimensions are within bounds.
    """
    lon_span = max_lon - min_lon
    lat_span = max_lat - min_lat
    if lon_span <= MAX_AOI_DEG and lat_span <= MAX_AOI_DEG:
        return min_lon, min_lat, max_lon, max_lat
    cx = (min_lon + max_lon) / 2
    cy = (min_lat + max_lat) / 2
    half = MAX_AOI_DEG / 2
    return cx - half, cy - half, cx + half, cy + half


def download_image(image, aoi, incident_id, filename, scale=10, max_retries=5):
    """Download a single GeoTIFF from GEE with retry on 429 rate limits."""
    os.makedirs(f"{DOWNLOAD_DIR}/incident_{incident_id}", exist_ok=True)
    filepath = f'{DOWNLOAD_DIR}/incident_{incident_id}/{filename}.tif'
    for attempt in range(1, max_retries + 1):
        try:
            url = image.getDownloadURL({
                'scale': scale,
                'region': aoi,
                'format': 'GeoTIFF',
                'crs': 'EPSG:4326',
            })
            response = requests.get(url, stream=True, timeout=300)
            if response.status_code == 429:
                wait = 15 * attempt
                print(f"  429 on {filename}, retry {attempt}/{max_retries} after {wait}s")
                time.sleep(wait)
                continue
            response.raise_for_status()
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"Downloaded: {filepath}")
            return
        except Exception as e:
            if attempt == max_retries:
                print(f"Failed to download {filename}: {e}")
                return
            wait = 15 * attempt
            print(f"  error on {filename}, retry {attempt}/{max_retries} after {wait}s: {e}")
            time.sleep(wait)


def best_scene(collection, label):
    """Return the least-cloudy single image from a filtered S2 collection.
    No temporal blending — picks one overpass, then mosaics MGRS tiles spatially.
    """
    count = collection.size().getInfo()
    if count == 0:
        print(f"  {label}: no scenes found.")
        return None
    return collection.sort('CLOUDY_PIXEL_PERCENTAGE').first()


def get_single_sar_image(s1_collection, label):
    """Return the earliest available S1 image from a filtered collection."""
    count = s1_collection.size().getInfo()
    if count == 0:
        print(f"  SAR {label}: no scenes found.")
        return None
    return s1_collection.sort('system:time_start').first()


def submit_landslide_export(incident_id, pre_days=180, post_days=45, include_sar=True):
    """Download all imagery for a single landslide incident.

    Downloads:
      - Sentinel-2 before/after (13 bands: B1-B12 + SCL, 10m)
      - DEM slope and aspect (30m)
      - Sentinel-1 GRD pre/post (VV/VH, 10m, same orbit direction)

    See landslide_workflow.md Stage 0.2-0.4 for the methodology.
    """
    incident_id = int(incident_id)
    row = df[df['id'] == incident_id]
    if row.empty:
        print(f"ID {incident_id} not found in CSV.")
        return
    row = row.iloc[0]
    incident_date = row['incident_on']

    c_min_lon, c_min_lat, c_max_lon, c_max_lat = clamp_aoi(
        row['min_lon'], row['min_lat'], row['max_lon'], row['max_lat'])
    aoi = ee.Geometry.Rectangle([c_min_lon, c_min_lat, c_max_lon, c_max_lat])

    # Temporal windows: pre-event up to 5 days before, post-event from 5 days after
    before_start = (incident_date - pd.DateOffset(days=pre_days)).strftime('%Y-%m-%d')
    before_end   = (incident_date - pd.DateOffset(days=5)).strftime('%Y-%m-%d')
    after_start  = (incident_date + pd.DateOffset(days=5)).strftime('%Y-%m-%d')
    after_end    = (incident_date + pd.DateOffset(days=post_days)).strftime('%Y-%m-%d')

    print(f"\nProcessing ID {incident_id}: {row['title']}")

    # ---- Sentinel-2: fetch and pick best scenes ----
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 70))
            .map(mask_s2_clouds))

    bands = ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B11','B12','SCL']

    before_img = best_scene(s2.filterDate(before_start, before_end), 'before')
    after_img  = best_scene(s2.filterDate(after_start, after_end), 'after')

    if before_img is None or after_img is None:
        print(f"Skipping ID {incident_id}  -  missing pre or post scene.")
        return

    print(f"  before date: {before_img.date().format().getInfo()}")
    print(f"  after  date: {after_img.date().format().getInfo()}")

    # Download pre/post S2
    download_image(before_img.select(bands).clip(aoi), aoi, incident_id,
                   f'incident_{incident_id}_before', scale=10)
    download_image(after_img.select(bands).clip(aoi), aoi, incident_id,
                   f'incident_{incident_id}_after', scale=10)

    # ---- DEM derivatives (Stage 0.4) ----
    dem = ee.Image('USGS/SRTMGL1_003')
    slope = ee.Terrain.slope(dem).clip(aoi)
    aspect = ee.Terrain.aspect(dem).clip(aoi)
    download_image(slope, aoi, incident_id, f'incident_{incident_id}_slope', scale=30)
    download_image(aspect, aoi, incident_id, f'incident_{incident_id}_aspect', scale=30)

    # ---- Sentinel-1 GRD (Stage 0.3) ----
    # Critical: same orbit direction (ASCENDING/DESCENDING) for pre and post
    if include_sar:
        try:
            s1_base = (ee.ImageCollection('COPERNICUS/S1_GRD')
                        .filterBounds(aoi)
                        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                        .filter(ee.Filter.eq('instrumentMode', 'IW')))
            sar_pre_start = (incident_date - pd.DateOffset(days=45)).strftime('%Y-%m-%d')
            sar_post_end  = (incident_date + pd.DateOffset(days=post_days)).strftime('%Y-%m-%d')
            s1_pre  = s1_base.filterDate(sar_pre_start, before_end)
            s1_post = s1_base.filterDate(after_start, sar_post_end)

            pre_pass  = s1_pre.aggregate_array('orbitProperties_pass').getInfo()
            post_pass = s1_post.aggregate_array('orbitProperties_pass').getInfo()
            common_passes = set(pre_pass) & set(post_pass)
            if not common_passes:
                raise ValueError('No common orbit pass between pre and post S1')
            orbit = sorted(common_passes)[0]

            s1_img_pre  = get_single_sar_image(
                s1_pre.filter(ee.Filter.eq('orbitProperties_pass', orbit)), 'pre')
            s1_img_post = get_single_sar_image(
                s1_post.filter(ee.Filter.eq('orbitProperties_pass', orbit)), 'post')

            if s1_img_pre is not None and s1_img_post is not None:
                sar_bands = ['VV', 'VH']
                download_image(s1_img_pre.select(sar_bands).clip(aoi), aoi, incident_id,
                               f'incident_{incident_id}_sar_pre', scale=10)
                download_image(s1_img_post.select(sar_bands).clip(aoi), aoi, incident_id,
                               f'incident_{incident_id}_sar_post', scale=10)
                print(f"  SAR pre date: {s1_img_pre.date().format().getInfo()}")
                print(f"  SAR post date: {s1_img_post.date().format().getInfo()}")
        except Exception as e:
            print(f"  SAR fetch skipped: {e}")

## Step 3: Process missing incidents and upload

We process incidents concurrently (2 workers to avoid GEE rate limits), download
all GeoTIFFs locally, then upload to HuggingFace in batches. After each successful
upload, the local incident folder is cleaned up to free disk space.

In [4]:
# %% [code]
def flush_uploads():
    """Upload all locally downloaded GeoTIFFs to HuggingFace, then clean up."""
    tif_count = len(glob.glob(f"{DOWNLOAD_DIR}/**/*.tif", recursive=True))
    if tif_count == 0:
        print("No new GeoTIFFs to upload.")
        return
    print(f"Uploading {tif_count} GeoTIFFs to {repo_id}...")
    try:
        api.upload_folder(
            folder_path=DOWNLOAD_DIR,
            repo_id=repo_id,
            repo_type="dataset",
            revision=DATASET_REVISION,
            allow_patterns="*.tif",
        )
        print(f"Upload successful. Cleaning up {DOWNLOAD_DIR}...")
        for subdir in glob.glob(f"{DOWNLOAD_DIR}/*/"):
            shutil.rmtree(subdir)
        print(f"Cleaned {DOWNLOAD_DIR}")
    except Exception as e:
        print(f"!!! Upload failed, keeping local files for retry: {e} !!!")


def process_incident(inc_id):
    """Wrapper to download a single incident's imagery from GEE."""
    submit_landslide_export(inc_id)
    return inc_id


# --------------------------------------------------------------------
# Main loop: only process incidents missing from HuggingFace
# --------------------------------------------------------------------
if not missing_ids:
    print("\n=== No missing incidents. Dataset is fully synced! ===")
else:
    print(f"\n=== Processing {len(missing_ids)} missing incidents ===")
    upload_count = 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(process_incident, inc_id): inc_id for inc_id in missing_ids}
        for future in as_completed(futures):
            inc_id = futures[future]
            try:
                future.result()
            except Exception as e:
                print(f"Incident {inc_id} failed: {e}")
            upload_count += 1
            if upload_count % upload_batch == 0:
                flush_uploads()

    # Upload any remaining incidents that didn't fill a full batch
    if upload_count % upload_batch != 0:
        flush_uploads()

    print(f"\n=== Sync complete. Processed {upload_count} new incidents. ===")


=== Processing 1601 missing incidents ===

Processing ID 37370: Landslide at Katari Municipality-12

Processing ID 37656: Landslide at Dharche Rural Municipality-3


/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  after: no scenes found.
Skipping ID 37370  -  missing pre or post scene.

Processing ID 37722: Landslide at Narayan Municipality-11
  after: no scenes found.
Skipping ID 37656  -  missing pre or post scene.

Processing ID 37814: Landslide at Tripurasundari Municipality-6
  after: no scenes found.
Skipping ID 37814  -  missing pre or post scene.

Processing ID 38007: Landslide at Sisne Rural Municipality-7
  after: no scenes found.
Skipping ID 37722  -  missing pre or post scene.

Processing ID 38048: Landslide at Bhimeshwor Municipality-3
  after: no scenes found.
Skipping ID 38007  -  missing pre or post scene.

Processing ID 38053: Landslide at Makalu Rural Municipality-3
  after: no scenes found.
Skipping ID 38048  -  missing pre or post scene.

Processing ID 38054: Landslide at Makalu Rural Municipality-3
  after: no scenes found.
Skipping ID 38053  -  missing pre or post scene.

Processing ID 38133: Landslide at Thawang Rural Municipality-3
  after: no scenes found.
Skipping ID 

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

Downloaded: /kaggle/working/downloads/incident_59452/incident_59452_aspect.tif
Downloaded: /kaggle/working/downloads/incident_60010/incident_60010_before.tif
Downloaded: /kaggle/working/downloads/incident_59452/incident_59452_sar_pre.tif
Downloaded: /kaggle/working/downloads/incident_60010/incident_60010_after.tif
Downloaded: /kaggle/working/downloads/incident_60010/incident_60010_slope.tif
Downloaded: /kaggle/working/downloads/incident_59452/incident_59452_sar_post.tif
  SAR pre date: 2022-05-02T12:22:27
  SAR post date: 2022-06-19T12:22:30

Processing ID 60014: Landslide at Borsaawang, Madi Rural Municipality-3
Downloaded: /kaggle/working/downloads/incident_60010/incident_60010_aspect.tif
  before date: 2022-03-09T05:20:59
  after  date: 2022-08-11T05:20:58
Downloaded: /kaggle/working/downloads/incident_60010/incident_60010_sar_pre.tif
Downloaded: /kaggle/working/downloads/incident_60014/incident_60014_before.tif
Downloaded: /kaggle/working/downloads/incident_60010/incident_60010_sar

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Downloaded: /kaggle/working/downloads/incident_74732/incident_74732_sar_pre.tif
Downloaded: /kaggle/working/downloads/incident_74732/incident_74732_sar_post.tif
  SAR pre date: 2024-08-19T12:22:35
  SAR post date: 2024-10-06T12:22:37
Upload successful. Cleaning up /kaggle/working/downloads...
Cleaned /kaggle/working/downloads
No new GeoTIFFs to upload.

=== Sync complete. Processed 1601 new incidents. ===
